In [1]:
%cd '/home/mojtaba.kananisarche/'

/home/mojtaba.kananisarche


In [63]:
from Zh.src.utils.load_data import load_data
import numpy as np
import pandas as pd
from typing import List, Optional, Tuple
from typing import Optional
from Zh.src.utils.file_folder import create_folders
import os
import json
import pandas as pd
import dataframe_image as dfi
import asyncio
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
)
from collections.abc import Iterable

In [9]:
def pad_sequences(sequences, maxlen=None, dtype='int32', padding='pre', truncating='pre', value=0):
    """
    Pads sequences to the same length.

    Args:
        sequences: List of sequences (lists or arrays of varying lengths).
        maxlen: Maximum length of all sequences after padding. If None, it uses the length of the longest sequence.
        dtype: Type of the output sequences.
        padding: 'pre' or 'post', pad either before or after each sequence.
        truncating: 'pre' or 'post', remove values from sequences longer than maxlen at the beginning or end.
        value: Padding value.

    Returns:
        A numpy array of shape (number_of_sequences, maxlen)
    """
    lengths = [len(s) for s in sequences]

    if maxlen is None:
        maxlen = max(lengths)

    padded_sequences = np.full((len(sequences), maxlen), value, dtype=dtype)

    for idx, seq in enumerate(sequences):
        if len(seq) > maxlen:
            if truncating == 'pre':
                trunc = seq[-maxlen:]
            elif truncating == 'post':
                trunc = seq[:maxlen]
            else:
                raise ValueError("truncating must be 'pre' or 'post'")
        else:
            trunc = seq

        if padding == 'pre':
            padded_sequences[idx, -len(trunc):] = trunc
        elif padding == 'post':
            padded_sequences[idx, :len(trunc)] = trunc
        else:
            raise ValueError("padding must be 'pre' or 'post'")

    return padded_sequences


In [6]:
sequences = [[1, 2, 3], [4, 5], [6]]
print(pad_sequences(sequences, maxlen=4, padding='pre', value=0))

[[0 1 2 3]
 [0 0 4 5]
 [0 0 0 6]]


In [8]:
path_to_data = "/work/long_lab/Mojtaba/data/UTA/1.3.3-preprocessed"
preprocessed_data = load_data(
    path_to_data=path_to_data,
    file_names=["facenet_preprocessed_alternativemapping_croppedonly.csv"],
    nrows=100000,
)


2025-04-17 04:25:15,573 - INFO - Concatenated file: facenet_preprocessed_alternativemapping_croppedonly.csv, current data shape: (99999, 135)


In [ ]:
def reshape_data_rnn(
    df: pd.DataFrame,
    feature_columns: List[str],
    slice_size: int,
    step_size: Optional[int],
    groupby_column: str = "file_name",
    sortby_column: str = "frame",
    target_column: str = "target",
    discard_threshold_ratio: float = 0.6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Reshape data for RNN input by grouping, slicing, and padding sequences.

    Args:
        df (pd.DataFrame)              : Input dataframe.
        feature_columns (List[str])    : List of feature column names.
        slice_size (int)               : Maximum number of slices in each chunk.
        step_size (int)                : Step size for moving to the next chunk.
        groupby_column (str)           : Column to group by (e.g., video or file name).
        sortby_column (str)            : Column to sort within each group (e.g., frame index).
        target_column (str)            : Column containing target labels.
        discard_threshold_ratio (float): Last slice in each video will be discarded if the number of its frames is less than
                                         discard_threshold_ratio*max_slice and will be post padded with zero and kept otherwise.
    Returns:
        Tuple[np.ndarray, np.ndarray]: Feature sequences (X) and corresponding labels (y).
    """
    step_size = step_size or slice_size
    X, y, g = [], [], []

    for group_identifier, group in df.groupby(groupby_column):
        # Sort by the specified column to maintain temporal order
        group = group.sort_values(by=sortby_column, ascending=True)

        y_group = group[target_column].unique()
        assert (
            len(y_group) == 1
        ), f"Expected one unique target value, but found {len(y_group)}"
        y_group = y_group[0]

        # Extract features as a list of lists
        slices = group[feature_columns].values.tolist()
        # Create chunks with the specified step size
        for start_idx in range(0, len(slices) - slice_size + 1, step_size):
            chunk = slices[start_idx : start_idx + slice_size]
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

        discard_threshold_frame = int(discard_threshold_ratio * slice_size)
        if len(slices) - (start_idx + slice_size) > discard_threshold_frame:
            chunk = slices[start_idx + slice_size : len(slices)]
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

    # Pad sequences to ensure uniform length
    X_padded = pad_sequences(sequences=X, maxlen=slice_size, padding="post", dtype="float32")

    return np.array(X_padded), np.array(y), g


In [46]:
columns_to_drop = [
    "file_name",
    "file_name_1",
    "type",
    "frame",
    "fold",
    "subject_id",
]
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
columns_to_drop_1 = [
    column_name
    for column_name in columns_to_drop
    if column_name not in [groupby_column, sortby_column, target_column]
]

preprocessed_data_2 = preprocessed_data.drop(columns=columns_to_drop_1)

feature_columns = [
    column_name
    for column_name in preprocessed_data_2.columns
    if column_name not in columns_to_drop
]

slice_size = 4500
step_size = 3000
discard_threshold_ratio = 0.7

In [47]:
X_train, y_train, group_identifiers_train = reshape_data_rnn(
    df=preprocessed_data,
    feature_columns=feature_columns,
    slice_size=slice_size,
    step_size=step_size,
    discard_threshold_ratio=discard_threshold_ratio,
    groupby_column=groupby_column,
    sortby_column=sortby_column,
    target_column=target_column,
)

In [48]:
for i, j in zip(y_train, group_identifiers_train):
    print(f"{i} - {j}")

6.5 - 1_02_5_30fps_cropped.csv
6.5 - 1_02_5_30fps_cropped.csv
6.5 - 1_02_5_30fps_cropped.csv
6.5 - 1_02_5_30fps_cropped.csv
6.5 - 1_02_5_30fps_cropped.csv
2.0 - 1_04_0_30fps_cropped.csv
2.0 - 1_04_0_30fps_cropped.csv
2.0 - 1_04_0_30fps_cropped.csv
2.0 - 1_04_0_30fps_cropped.csv
2.0 - 1_04_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 2_20_0_30fps_cropped.csv
2.0 - 4_38_0_30fps_cropped.csv
2.0 - 4_38_0_30fps_cropped.csv
2.0 - 4_38_0_30fps_cropped.csv
2.0 - 4_38_0_30fps_cropped.csv
2.0 - 4_38_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_55_0_30fps_cropped.csv
2.0 - 5_56_0_30fps_cropped.csv


In [4]:
async def table_results(
results_directory_path : str,
output_directory_path  : Optional[str] = None,
output_file_path_prefix: Optional[str] = 'results',
cmap                   : Optional[str] = "Greys",
reverse_cmap           : Optional[bool] = False,
maximum_tries          : Optional[int] = 5,
)->None: 
    
    KEYS_TO_COLLECT       = ['index', 'test', 'val', 'train_report', 'val_report', 'test_report']
    COLORS                = ['Accent','Blues','BrBG','BuGn','BuPu','CMRmap','Dark2','GnBu','Greens',
                             'Greys','OrRd','Oranges','PRGn','Paired','Pastel1','Pastel2','PiYG',
                             'PuBuGn','PuBu','PuOr','PuRd','Purples','RdBu','RdGy','RdPu','RdYlBu',
                             'RdYlGn','Reds','Set1','Set2','Set3','Spectral','Wistia','YlGnBu','YlGn',
                             'YlOrBr','YlOrRd','afmhot','autumn','binary','bone','brg','bwr','cividis',
                             'cool','coolwarm','copper','cubehelix','flag','gist_earth','gist_gray',
                             'gist_heat','gist_ncar','gist','gist_rainbow','gist_stern','gist_yarg',
                             'gnuplot2','gnuplot','gray','hot','hsv','inferno','jet','magma','nipy_spectral'
                             ,'ocean','pink','plasma','prism','rainbow','seismic','spring','summer','tab10',
                             'tab20','tab20b','tab20c','terrain','turbo','twilight','twilight_shifted','viridis','winter']

    output_directory_path = output_directory_path or results_directory_path
    create_folders(output_directory_path)
    output_file_name = None

    i = 0
    while output_file_name is None:
        i += 1
        if not os.path.exists(os.path.join(output_directory_path, f"{output_file_path_prefix}_{i}.png")):
            output_file_name = f"{output_file_path_prefix}_{i}.png"
    
    output_file_path = os.path.join(output_directory_path, output_file_name)

    data = []
    for file_name in os.listdir(results_directory_path):
        if file_name.endswith('.json'):
            file_path = os.path.join(results_directory_path, file_name)
            with open(file=file_path, mode='r') as json_file:
                j = json.load(json_file)
                try:
                    data.append(
                        {
                            k:v for k, v in j.items() if k in KEYS_TO_COLLECT
                        }
                    )
                except KeyError as e:
                    print("Unable")
                    continue


    data = pd.DataFrame(data = data).dropna(how='all')
    try:
        data.index = data['index'].astype(int).values
        data       = data.sort_index()
        data       = data.drop('index', axis=1)
    except:
        print("No Index")
        pass

    folded_by_column = list(data['test'][0].keys())[0]

    multi_index = pd.MultiIndex.from_tuples([
    (f'Split ({folded_by_column})', 'Test'),
    (f'Split ({folded_by_column})', 'Validation'),
    ('Report', 'Train'),
    ('Report', 'Validation'),
    ('Report', 'Test'),
    ])
    data.columns = multi_index

    data[(f'Split ({folded_by_column})', 'Test')] = data[(f'Split ({folded_by_column})', 'Test')].apply(lambda x: x[folded_by_column])
    data[(f'Split ({folded_by_column})', 'Validation')]  = data[(f'Split ({folded_by_column})', 'Validation')].apply(lambda x: x[folded_by_column])

    data[('MAE', 'Train')] = data[('Report', 'Train')].apply(lambda x: round(x['overall']['mae'], 3))
    data[('MAE', 'Validation')]   = data[('Report', 'Validation')].apply(lambda x: round(x['overall']['mae'], 3))
    data[('MAE', 'Test')]  = data[('Report', 'Test')].apply(lambda x: round(x['overall']['mae'], 3))

    data[('Accuracy', 'Train')] = data[('Report', 'Train')].apply(lambda x: round(x['overall']['accuracy'], 3))
    data[('Accuracy', 'Validation')]   = data[('Report', 'Validation')].apply(lambda x: round(x['overall']['accuracy'], 3))
    data[('Accuracy', 'Test')]  = data[('Report', 'Test')].apply(lambda x: round(x['overall']['accuracy'], 3))

    data = data.drop('Report', axis=1, level=0)
    data.columns.names = [None, 'Index']

    avg = data.mean(numeric_only=True)
    data.loc['Average'] = avg
    data = data.fillna('-')

    cmaps = [cmap, f"{cmap}_r"] if reverse_cmap else [f"{cmap}_r", cmap]

    data_styled = data.style.background_gradient(subset=[('Accuracy', 'Train'), 
                                                         ('Accuracy', 'Validation'), 
                                                         ('Accuracy', 'Test')], axis=None, vmin=0, vmax=1, cmap=cmaps[0]) \
                        .background_gradient(subset=[('MAE', 'Train'), 
                                                     ('MAE', 'Validation'), 
                                                     ('MAE', 'Test')], vmin=0, vmax=1, axis=None, cmap=cmaps[1]) \
                    .format(precision=3) \
                    .set_table_styles(
                        [{'selector': 'th', 'props': [('text-align', 'center')]},  # Center align column names
                            {'selector': 'td', 'props': [('text-align', 'center')]},  # Center align values
                            {'selector': 'thead th', 'props': [('text-align', 'center')]},  # Center align indices
                            {'selector': 'td:nth-child(2)', 'props': [('border-left', '4px solid black')]},  # Add line between index and Test
                            {'selector': 'td:nth-child(4)', 'props': [('border-left', '4px solid black')]},  # Add line between val and train_mae
                            {'selector': 'td:nth-child(7)', 'props': [('border-left', '4px solid black')]},  # Add line between test_mae and train_accuracy
                            {'selector': 'td:nth-child(2)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(3)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(4)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(5)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(6)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(7)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(8)', 'props': [('width', '5em')]},
                            {'selector': 'td:nth-child(9)', 'props': [('width', '5em')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(3)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(1) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(2)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(4)', 'props': [('border-left', '4px solid black')]},
                            {'selector': 'thead tr:nth-child(2) th:nth-child(7)', 'props': [('border-left', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('border-top', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('border-bottom', '4px solid black')]},
                            {'selector': f'tr:nth-child({data.shape[0]})', 'props': [('font-weight', 'bold')]}
                            ]  # Add line above the last horizontal row
                    ) \
                    .set_properties(**{'text-align': 'center'})  # Ensure all text is center aligned

    for i in range(maximum_tries):
        try:
            await dfi.export_async(obj=data_styled, filename=output_file_path, table_conversion='playwright')
            print(f"Try Number {i} Succeeded.")
            return
        except:
            print(f"Try Number {i} Failed.")


In [98]:
from pickle import NONE


async def table_results(
results_directory_path : str,
unique_classes         : List[float],
output_directory_path  : Optional[str] = None,
output_file_path_prefix: Optional[str] = 'results',
index                  : Optional[int] = -1,
split                  : Optional[str] = 'all', #train, val, test, all
cmap                   : Optional[str] = "Greys",
reverse_cmap           : Optional[bool] = False,
maximum_tries          : Optional[int] = 5,
corner_text_color      : Optional[str] = 'white',
corner_bg_color        : Optional[str] = 'IndianRed',
pred_true_text_color   : Optional[str]  = 'black',
pred_true_bg_color     : Optional[str] = 'silver',
labels_bg_color        : Optional[str] = 'beige',
labels_text_color      : Optional[str] = 'black',
height                 : int           = 4,
width                  : Optional[int] = None,
): 
    
    width = width or height

    thresholds = []
    unique_classes = sorted(unique_classes)
    for i in range(len(unique_classes) - 1):
        thresholds.append((unique_classes[i] + unique_classes[i + 1]) / 2)

    KEYS_TO_COLLECT       = ['index', 'test', 'val', 'y_train', 'y_train_pred', 'y_val', 'y_val_pred', 'y_test', 'y_test_pred']

    output_directory_path = output_directory_path or results_directory_path
    create_folders(output_directory_path)
    output_file_name = None

    i = 0
    while output_file_name is None:
        i += 1
        if not os.path.exists(os.path.join(output_directory_path, f"{output_file_path_prefix}_{i}.png")):
            output_file_name = f"{output_file_path_prefix}_{i}.png"
    
    output_file_path = os.path.join(output_directory_path, output_file_name)

    data = []
    for file_name in os.listdir(results_directory_path):
        if file_name.endswith('.json'):
            file_path = os.path.join(results_directory_path, file_name)
            with open(file=file_path, mode='r') as json_file:
                j = json.load(json_file)
                try:
                    data.append(
                        {
                            k:v for k, v in j.items() if k in KEYS_TO_COLLECT
                        }
                    )
                except KeyError as e:
                    print("Unable")
                    continue


    data = pd.DataFrame(data = data).dropna(how='all')
    try:
        data.index = data['index'].astype(int).values
        data       = data.sort_index()
        data       = data.drop('index', axis=1)
    except:
        print("No Index")
        pass

    folded_by_column           = list(data['test'][0].keys())[0]

    def style(confusion_matrix_df
              ):
        
        confusion_matrix_df.columns = pd.MultiIndex.from_product(iterables=[['Predicted'], [str(round(float(i),1)) for i in unique_classes]])
        confusion_matrix_df.index = pd.MultiIndex.from_product(iterables=[['True'], [str(round(float(i),1)) for i in unique_classes]])

        styled = confusion_matrix_df.style.background_gradient(axis=1, cmap=cmap)\
                .set_table_styles([
                    {'selector': 'th', 
                     'props':[('height', f'{height}em'), 
                              ('width', f'{width}em'), 
                              ('text-align', 'center')]},
                    {'selector': 'td', 
                     'props':[('height', f'{height}em'), 
                              ('width', f'{width}em'), 
                              ('text-align', 'center')]},

                    {'selector': 'thead tr:nth-child(1) th:nth-child(1)',
                     'props': [('background-color', corner_bg_color), 
                               ('color', corner_text_color)]},

                    {'selector': 'th:nth-child(2)', 
                     'props': [('border-right', '1px solid black')]},
                    {'selector': 'td:nth-child(2)', 
                     'props': [('border-left', '1px solid black')]},

                    {'selector': 'thead tr:nth-child(1) th:nth-child(3)', 
                     'props': [('background-color', pred_true_bg_color), 
                               ('color', pred_true_text_color)]},
                    {'selector': 'tbody tr:nth-child(1) th:nth-child(1)', 
                     'props': [('background-color', pred_true_bg_color), 
                               ('color', pred_true_text_color)]},

                    {'selector': 'thead tr:nth-child(2) th:nth-child(3)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                    {'selector': 'thead tr:nth-child(2) th:nth-child(4)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                    {'selector': 'thead tr:nth-child(2) th:nth-child(5)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                    {'selector': 'tbody tr:nth-child(1) th:nth-child(2)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                    {'selector': 'tbody tr:nth-child(2) th:nth-child(1)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                    {'selector': 'tbody tr:nth-child(3) th:nth-child(1)', 
                     'props': [('background-color', labels_bg_color), 
                               ('color', labels_text_color)]},
                ])
        return styled

    styled_dfs = {}
    if index > 0:
        split = ['train', 'test', 'val'] if split == 'all' else [split]
        for s in split:
            y_true = [str(float(yt)) for yt in data[f'y_{s}'][index]]
            y_pred = [str(float(unique_classes[np.digitize(x, thresholds, right=False)])) for x in data[f'y_{s}_pred'][index]]
            confusion_matrix_df = pd.DataFrame(data = confusion_matrix(y_true = y_true, y_pred = y_pred))
            styled_df = style(confusion_matrix_df)
            styled_dfs[f'{s}'] = styled_df

    else:
        split = ['train', 'test', 'val'] if split == 'all' else [split]
        for s in split:
            y_true = [str(float(yt)) for yt in flatten(list(data[f'y_{s}']))]
            y_pred = [str(float(yp)) for yp in flatten(list(data[f'y_{s}_pred'].apply(lambda x: [unique_classes[np.digitize(i, thresholds, right=False)] for i in x])))]
            confusion_matrix_df = pd.DataFrame(data = confusion_matrix(y_true = y_true, y_pred = y_pred))
            styled_df = style(confusion_matrix_df)
            styled_dfs[f'{s}'] = styled_df

            # data['y_train_pred_class'] = data['y_train_pred'].apply(lambda x: [unique_classes[np.digitize(i, thresholds, right=False)] for i in x])
            # data['y_val_pred_class']   = data['y_val_pred'].apply(lambda x: [unique_classes[np.digitize(i, thresholds, right=False)] for i in x])
            # data['y_test_pred_class']  = data['y_test_pred'].apply(lambda x: [unique_classes[np.digitize(i, thresholds, right=False)] for i in x])


    

        

    #     data_cm_1 = pd.DataFrame(data = confusion_matrix(y_true = [str(float(i)) for i in data['y_train'][index]], 
    #                                                      y_pred = [str(float(i)) for i in data['y_train_pred_class'][index]]))
        
    #     data_cm_2 = pd.DataFrame(data = confusion_matrix(y_true = [str(float(i)) for i in data['y_val'][index]], 
    #                                                      y_pred = [str(float(i)) for i in data['y_val_pred_class'][index]]))
        
    #     data_cm_3 = pd.DataFrame(data = confusion_matrix(y_true = [str(float(i)) for i in data['y_test'][index]], 
    #                                                      y_pred = [str(float(i)) for i in data['y_test_pred_class'][index]]))
        
    # if split == 'all':
    #     pass
    

    # data_style_1 = style(data_cm_1)
    # data_style_2 = style(data_cm_2)
    # data_style_3 = style(data_cm_3)

    return styled_dfs


    for i in range(maximum_tries):
        try:
            await dfi.export_async(obj=data_styled, filename=output_file_path, table_conversion='playwright')
            print(f"Try Number {i} Succeeded.")
            return
        except:
            print(f"Try Number {i} Failed.")

In [104]:
data = await table_results(results_directory_path = '/home/mojtaba.kananisarche/Zh/models/2025-04-15|23:45:55|RNN_FACENET_5CV', unique_classes=[2,6.5,8.5], index=10, height=3, width=3, split='all')

In [105]:
data['train']

In [106]:
data['test']

In [107]:
data['val']

In [74]:
def flatten(lst):
    flat = []
    for item in lst:
        if isinstance(item, list):
            flat.extend(flatten(item))
        else:
            flat.append(item)
    return flat

In [73]:
flatten(list((data['y_train'])))

[2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 8.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 6.5,
 2.0,
 2.0,
 2.0

In [45]:
df1, df2, df3 = await table_results(results_directory_path = '/home/mojtaba.kananisarche/Zh/models/2025-04-15|23:45:55|RNN_FACENET_5CV', unique_classes=[2,6.5,8.5], index=1, height=3, width=3)

In [46]:
df3

In [159]:
b = dfi.export_async(obj=df2, filename='test5.png', table_conversion='playwright', corner='Train')
await b

In [ ]:
kwargs = 



{'exclude_styles': False, 'encoding': 'utf-8', 'doctype_html': False, 'corner': 'test'}

In [11]:
if 'corner' in kwargs.keys():
    print(1)

1


In [9]:
a = {'uuid': 'ecb01', 'table_styles': [], 'caption': None, 'head': [[{'type': 'th', 'value': '&nbsp;', 'class': 'blank', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '&nbsp;', 'class': 'blank level0', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col0', 'is_visible': True, 'display_value': 'Predicted', 'attributes': 'colspan="3"', 'id': 'level0_col0'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col1', 'is_visible': False, 'display_value': 'Predicted', 'attributes': '', 'id': 'level0_col1'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col2', 'is_visible': False, 'display_value': 'Predicted', 'attributes': '', 'id': 'level0_col2'}], [{'type': 'th', 'value': '&nbsp;', 'class': 'blank', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '&nbsp;', 'class': 'blank level1', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '2.0', 'class': 'col_heading level1 col0', 'is_visible': True, 'display_value': '2.0', 'attributes': '', 'id': 'level1_col0'}, {'type': 'th', 'value': '6.5', 'class': 'col_heading level1 col1', 'is_visible': True, 'display_value': '6.5', 'attributes': '', 'id': 'level1_col1'}, {'type': 'th', 'value': '8.5', 'class': 'col_heading level1 col2', 'is_visible': True, 'display_value': '8.5', 'attributes': '', 'id': 'level1_col2'}]], 'index_lengths': {(0, 0): 3, (1, 0): 1, (1, 1): 1, (1, 2): 1}, 'body': [[{'type': 'th', 'value': 'True', 'class': 'row_heading level0 row0', 'is_visible': True, 'display_value': 'True', 'attributes': 'rowspan="3"', 'id': 'level0_row0'}, {'type': 'th', 'value': '2.0', 'class': 'row_heading level1 row0', 'is_visible': True, 'display_value': '2.0', 'attributes': '', 'id': 'level1_row0'}, {'type': 'td', 'value': 233, 'class': 'data row0 col0', 'is_visible': True, 'attributes': '', 'display_value': '233', 'id': 'row0_col0'}, {'type': 'td', 'value': 0, 'class': 'data row0 col1', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row0_col1'}, {'type': 'td', 'value': 0, 'class': 'data row0 col2', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row0_col2'}], [{'type': 'th', 'value': 'True', 'class': 'row_heading level0 row1', 'is_visible': False, 'display_value': 'True', 'attributes': '', 'id': 'level0_row1'}, {'type': 'th', 'value': '6.5', 'class': 'row_heading level1 row1', 'is_visible': True, 'display_value': '6.5', 'attributes': '', 'id': 'level1_row1'}, {'type': 'td', 'value': 0, 'class': 'data row1 col0', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row1_col0'}, {'type': 'td', 'value': 246, 'class': 'data row1 col1', 'is_visible': True, 'attributes': '', 'display_value': '246', 'id': 'row1_col1'}, {'type': 'td', 'value': 0, 'class': 'data row1 col2', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row1_col2'}], [{'type': 'th', 'value': 'True', 'class': 'row_heading level0 row2', 'is_visible': False, 'display_value': 'True', 'attributes': '', 'id': 'level0_row2'}, {'type': 'th', 'value': '8.5', 'class': 'row_heading level1 row2', 'is_visible': True, 'display_value': '8.5', 'attributes': '', 'id': 'level1_row2'}, {'type': 'td', 'value': 0, 'class': 'data row2 col0', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row2_col0'}, {'type': 'td', 'value': 0, 'class': 'data row2 col1', 'is_visible': True, 'attributes': '', 'display_value': '0', 'id': 'row2_col1'}, {'type': 'td', 'value': 286, 'class': 'data row2 col2', 'is_visible': True, 'attributes': '', 'display_value': '286', 'id': 'row2_col2'}]], 'cellstyle': [{'props': [('background-color', '#000000'), ('color', '#f1f1f1'), ('text-align', 'center')], 'selectors': ['row0_col0', 'row1_col1', 'row2_col2']}, {'props': [('background-color', '#ffffff'), ('color', '#000000'), ('text-align', 'center')], 'selectors': ['row0_col1', 'row0_col2', 'row1_col0', 'row1_col2', 'row2_col0', 'row2_col1']}], 'cellstyle_index': [], 'cellstyle_columns': [], 'table_attributes': None, 'exclude_styles': False, 'encoding': 'utf-8', 'doctype_html': False}

In [18]:
i = 3
print(f"{list(a.keys())[i]} - {a[list(a.keys())[i]]}")

head - [[{'type': 'th', 'value': '&nbsp;', 'class': 'blank', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '&nbsp;', 'class': 'blank level0', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col0', 'is_visible': True, 'display_value': 'Predicted', 'attributes': 'colspan="3"', 'id': 'level0_col0'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col1', 'is_visible': False, 'display_value': 'Predicted', 'attributes': '', 'id': 'level0_col1'}, {'type': 'th', 'value': 'Predicted', 'class': 'col_heading level0 col2', 'is_visible': False, 'display_value': 'Predicted', 'attributes': '', 'id': 'level0_col2'}], [{'type': 'th', 'value': '&nbsp;', 'class': 'blank', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '&nbsp;', 'class': 'blank level1', 'is_visible': True, 'display_value': '&nbsp;'}, {'type': 'th', 'value': '2.0', 'class': 'col_heading level1 col0', 'is_vi

In [22]:
a[list(a.keys())[3]][0]

[{'type': 'th',
  'value': '&nbsp;',
  'class': 'blank',
  'is_visible': True,
  'display_value': '&nbsp;'},
 {'type': 'th',
  'value': '&nbsp;',
  'class': 'blank level0',
  'is_visible': True,
  'display_value': '&nbsp;'},
 {'type': 'th',
  'value': 'Predicted',
  'class': 'col_heading level0 col0',
  'is_visible': True,
  'display_value': 'Predicted',
  'attributes': 'colspan="3"',
  'id': 'level0_col0'},
 {'type': 'th',
  'value': 'Predicted',
  'class': 'col_heading level0 col1',
  'is_visible': False,
  'display_value': 'Predicted',
  'attributes': '',
  'id': 'level0_col1'},
 {'type': 'th',
  'value': 'Predicted',
  'class': 'col_heading level0 col2',
  'is_visible': False,
  'display_value': 'Predicted',
  'attributes': '',
  'id': 'level0_col2'}]

In [26]:
a[list(a.keys)[2]]

TypeError: 'builtin_function_or_method' object is not iterable

In [31]:
a['head'][0][0]['display_value'] = 'Test'

In [213]:
df1, df2, df3 = await table_results(results_directory_path = '/home/mojtaba.kananisarche/Zh/models/2025-04-15|23:45:55|RNN_FACENET_5CV', unique_classes=[2,6.5,8.5], index=1)

In [ ]:
df1.set_table_styles(
    [{'selector': 'thead tr:nth-child(1) th:nth-child(1)',
      'props': [('background-color', 'IndianRed'), ('color', 'white')]}]
)

In [215]:
df1.apply(
    lambda x: ['font-weight: bold' if v == max(x) else '' for v in x],
    subset=['Predicted']
)

In [ ]:
df1.apply()

In [116]:
df2.style.background_gradient(axis=1, cmap='Greys')

In [114]:
df3.style.background_gradient(axis=1, cmap='Greys')

In [94]:
df3.style.background_gradient(axis=1)

In [40]:
df.columns = pd.MultiIndex.from_tuples([('True', '2.0'),
            ('True', '6.5'),
            ('True', '8.5')])

In [57]:
df.style.background_gradient(axis=0)

In [23]:
df.style.background_gradient()\
    .format(precision=3)\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}, 
                       {'selector': 'td', 'props': [('text-align', 'center')]}]).set_properties(**{'text-align': 'center'})

In [43]:
classification_report(y_true = [str(i) for i in data['y_train'][0]], y_pred = [str(i) for i in data['y_train_pred_class'][0]], zero_division=False, output_dict=True)

{'2': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0},
 '2.0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 246.0},
 '6.5': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 256.0},
 '8.5': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 285.0},
 'accuracy': 0.6874205844980941,
 'macro avg': {'precision': 0.5,
  'recall': 0.5,
  'f1-score': 0.5,
  'support': 787.0},
 'weighted avg': {'precision': 0.6874205844980941,
  'recall': 0.6874205844980941,
  'f1-score': 0.6874205844980941,
  'support': 787.0}}

In [68]:
df = pd.DataFrame(data = confusion_matrix(y_true = [str(float(i)) for i in data['y_train'][0]], 
                 y_pred = [str(float(i)) for i in data['y_train_pred_class'][0]],
                 labels = [str(float(i)) for i in unique_classes]))
df.columns = pd.MultiIndex.from_product(iterables=[['True'], unique_classes])
df.index = pd.MultiIndex.from_product(iterables=[['Predicted'], unique_classes])

In [ ]:
df

True          
               2.0  6.5  8.5
Predicted 2.0  246    0    0
          6.5    0  256    0
          8.5    0    0  285

In [58]:
confusion_matrix(y_true = [str(float(i)) for i in data['y_train'][0]], 
                 y_pred = [str(float(i)) for i in data['y_train_pred_class'][0]],
                #  labels = ['2.0', '6.5', '8.5'])
                 labels = ['2.0', '8.5', '6.5'])

array([[246,   0,   0],
       [  0, 285,   0],
       [  0,   0, 256]])

In [54]:
set([str(i) for i in data['y_train_pred_class'][0]])

{'2', '6.5', '8.5'}

In [114]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
)

In [123]:
y_true = np.array([3,2,1,2,3,2,1,2,1,2,3])
y_pred = np.array([1,1,1,2,2,2,1,1,1,3,2]).flatten()
unique_classes = [1,2,3]

results = {}
thresholds = []

# Sort unique classes and calculate thresholds
unique_classes = sorted(unique_classes)
for i in range(len(unique_classes) - 1):
    thresholds.append((unique_classes[i] + unique_classes[i + 1]) / 2)

# Classify predictions
y_pred_classified = np.digitize(y_pred, thresholds, right=False)
y_pred_classified = np.array([unique_classes[i] for i in y_pred_classified])

class_results = {}
for cls in unique_classes:
    idx = y_true == cls
    if np.all(idx == False):
        class_results[cls] = {"mae": -1, "accuracy": -1}
        continue
    mae = mean_absolute_error(y_true[idx], y_pred[idx])
    accuracy = np.mean(y_pred_classified[idx] == y_true[idx])
    class_results[cls] = {"mae": mae, "accuracy": accuracy}

results["classes"] = class_results
# Calculate overall metrics
overall_mae = mean_absolute_error(y_true, y_pred)
overall_accuracy = np.mean(y_pred_classified == y_true)
classificatio_report_results = classification_report(y_true=y_true, y_pred=y_pred, zero_division=False, output_dict=True)
confusion_matrix_results = confusion_matrix(y_true=y_true, y_pred=y_pred)
results["overall"] = {"mae": overall_mae, "accuracy": overall_accuracy, "classificatio_report":classificatio_report_results, "confusion_matrix":confusion_matrix_results}


y_pred, results

(array([1, 1, 1, 2, 2, 2, 1, 1, 1, 3, 2]),
 {'classes': {1: {'mae': 0.0, 'accuracy': np.float64(1.0)},
   2: {'mae': 0.6, 'accuracy': np.float64(0.4)},
   3: {'mae': 1.3333333333333333, 'accuracy': np.float64(0.0)}},
  'overall': {'mae': 0.6363636363636364,
   'accuracy': np.float64(0.45454545454545453),
   'classificatio_report': {'1': {'precision': 0.5,
     'recall': 1.0,
     'f1-score': 0.6666666666666666,
     'support': 3.0},
    '2': {'precision': 0.5,
     'recall': 0.4,
     'f1-score': 0.4444444444444444,
     'support': 5.0},
    '3': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0},
    'accuracy': 0.45454545454545453,
    'macro avg': {'precision': 0.3333333333333333,
     'recall': 0.4666666666666666,
     'f1-score': 0.3703703703703704,
     'support': 11.0},
    'weighted avg': {'precision': 0.36363636363636365,
     'recall': 0.45454545454545453,
     'f1-score': 0.38383838383838387,
     'support': 11.0}},
   'confusion_matrix': array([[3, 0, 0],
  

In [121]:
results['overall']['classificatio_report']

{'1': {'precision': 0.3333333333333333,
  'recall': 0.5,
  'f1-score': 0.4,
  'support': 2.0},
 '2': {'precision': 0.25,
  'recall': 0.5,
  'f1-score': 0.3333333333333333,
  'support': 2.0},
 '3': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0},
 '4': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0},
 '5': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0},
 '6': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0},
 'accuracy': 0.18181818181818182,
 'macro avg': {'precision': 0.09722222222222221,
  'recall': 0.16666666666666666,
  'f1-score': 0.12222222222222223,
  'support': 11.0},
 'weighted avg': {'precision': 0.10606060606060605,
  'recall': 0.18181818181818182,
  'f1-score': 0.13333333333333333,
  'support': 11.0}}

In [105]:
results['overall']

KeyError: 'overall'

In [89]:
class_results

{'a': {}}

In [84]:
confusion_matrix(y_true=y_true, y_pred=y_pred,)

array([[3, 1],
       [1, 3]])